In [1]:
import pandas as pd
import numpy as np

import joblib
import json

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv(
    "../datasets/cleaned/startup_info.csv",
    low_memory=False
)

print(df.shape)

(50000, 127)


In [3]:
def funding_label(score):

    if score < 35:
        return "Low"

    elif score < 60:
        return "Medium"

    else:
        return "High"


df["funding_strength_label"] = (
    df["funding_strength_score"]
    .apply(funding_label)
)

print(
    df["funding_strength_label"]
    .value_counts()
)

funding_strength_label
Medium    25026
High      14656
Low       10318
Name: count, dtype: int64


In [4]:
features = [

    "total_funding",

    "latest_funding",

    "funding_round_count",

    "investor_count",

    "top_tier_investor_count",

    "average_round_size",

    "valuation",

    "burn_rate",

    "runway_months"

]

X = df[features]

y = df["funding_strength_label"]

In [5]:
le = LabelEncoder()

y = le.fit_transform(y)

print(
    dict(
        zip(
            le.classes_,
            range(
                len(le.classes_)
            )
        )
    )
)

{'High': 0, 'Low': 1, 'Medium': 2}


In [6]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

In [7]:
model = RandomForestClassifier(

    n_estimators=300,

    max_depth=12,

    random_state=42,

    n_jobs=-1

)

model.fit(
    X_train,
    y_train
)

RandomForestClassifier(max_depth=12, n_estimators=300, n_jobs=-1,
                       random_state=42)

In [8]:
preds = model.predict(X_test)

acc = accuracy_score(
    y_test,
    preds
)

print(
    "Accuracy:",
    round(acc*100,2),
    "%"
)

print(
    classification_report(
        y_test,
        preds
    )
)

Accuracy: 97.06 %
              precision    recall  f1-score   support

           0       0.98      0.97      0.98      2931
           1       0.99      0.94      0.96      2064
           2       0.96      0.99      0.97      5005

    accuracy                           0.97     10000
   macro avg       0.98      0.96      0.97     10000
weighted avg       0.97      0.97      0.97     10000



In [9]:
importance = pd.DataFrame({

    "Feature": features,

    "Importance":
        model.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

print(
    importance
)

                   Feature  Importance
3           investor_count    0.358963
4  top_tier_investor_count    0.349927
2      funding_round_count    0.226774
0            total_funding    0.014648
6                valuation    0.010260
5       average_round_size    0.010137
7                burn_rate    0.009926
1           latest_funding    0.009860
8            runway_months    0.009505


In [11]:
joblib.dump(

    model,

    "../models/funding_strength_model/funding_strength_model.pkl"

)

joblib.dump(

    le,

    "../models/funding_strength_model/label_encoder.pkl"

)

print("Funding Strength Model Saved ✅")

Funding Strength Model Saved ✅


In [12]:
metadata = {

    "model_name":
        "Funding Strength Model",

    "algorithm":
        "Random Forest",

    "features":
        features,

    "classes":
        list(
            le.classes_
        )

}

with open(

    "../models/funding_strength_model/metadata.json",

    "w"

) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

In [13]:
df.to_csv(
    "../datasets/cleaned/startup_info.csv",
    index=False
)

print("Dataset Saved ✅")

Dataset Saved ✅


In [14]:
print(df["funding_strength_label"].value_counts())

print("Accuracy:", acc)

print(importance)

funding_strength_label
Medium    25026
High      14656
Low       10318
Name: count, dtype: int64
Accuracy: 0.9706
                   Feature  Importance
3           investor_count    0.358963
4  top_tier_investor_count    0.349927
2      funding_round_count    0.226774
0            total_funding    0.014648
6                valuation    0.010260
5       average_round_size    0.010137
7                burn_rate    0.009926
1           latest_funding    0.009860
8            runway_months    0.009505
